In [1]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from dotenv import load_dotenv, find_dotenv
import requests
import os

from pydantic import BaseModel, Field
from typing import Literal
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate

from tqdm import tqdm

from langchain_openai import ChatOpenAI
import json

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score

from typing import Tuple, Union, Dict

In [2]:

@dataclass
class KNNState:
    df: pd.DataFrame
    feature_cols: List[str]
    medians: pd.Series
    scaler: StandardScaler
    feats_imputed: np.ndarray
    X_scaled: np.ndarray
    nn: NearestNeighbors
    excel_offset: int
    # NEW: class-specific subsets
    label_col: str
    pos_indices: np.ndarray         # global row indices where IEC_HS == 1
    neg_indices: np.ndarray         # global row indices where IEC_HS == 0
    pos_X: Optional[np.ndarray]     # X_scaled for IEC_HS == 1
    neg_X: Optional[np.ndarray]     # X_scaled for IEC_HS == 0
    pos_nn: Optional[NearestNeighbors]
    neg_nn: Optional[NearestNeighbors]

In [36]:
def build_state(
    df: pd.DataFrame,
    use_cols: Optional[List[str]] = None,
    excel_header_on_row1: bool = True,
    label_col: str = "Outcome",
    zero_as_missing: bool = True,
) -> KNNState:
    """
    Pima Indians Diabetes setup:
      - label_col defaults to 'Outcome'
      - features = all columns except label (unless use_cols is provided)
      - optionally treat zeros in some medical fields as missing before median impute
    """
    if label_col not in df.columns:
        raise ValueError(f"Required label column '{label_col}' not found in DataFrame.")

    # Choose features
    feature_cols = use_cols if use_cols is not None else [c for c in df.columns if c != label_col]

    # Coerce to numeric
    feats = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    # Optionally treat zeros as missing for typical Pima fields
    if zero_as_missing:
        zero_missing = {"glucose", "bloodpressure", "skinthickness", "insulin", "bmi"}
        for c in feature_cols:
            if c.lower() in zero_missing:
                feats.loc[feats[c] == 0, c] = np.nan

    # Median impute
    medians = feats.median(numeric_only=True)
    feats = feats.fillna(medians)
    feats_imputed = feats.values

    # Scale & global NN
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(feats_imputed)
    nn = NearestNeighbors(metric="euclidean")
    nn.fit(X_scaled)

    # Class-specific subsets (Outcome == 1 vs 0)
    labels = df[label_col].values
    pos_mask = (labels == 1)
    neg_mask = (labels == 0)

    pos_indices = np.where(pos_mask)[0]
    neg_indices = np.where(neg_mask)[0]

    pos_X = X_scaled[pos_indices] if len(pos_indices) > 0 else None
    neg_X = X_scaled[neg_indices] if len(neg_indices) > 0 else None

    pos_nn = None
    neg_nn = None
    if pos_X is not None and len(pos_indices) > 0:
        pos_nn = NearestNeighbors(metric="euclidean")
        pos_nn.fit(pos_X)
    if neg_X is not None and len(neg_indices) > 0:
        neg_nn = NearestNeighbors(metric="euclidean")
        neg_nn.fit(neg_X)

    excel_offset = 2 if excel_header_on_row1 else 1  # harmless; kept for API compatibility
    return KNNState(
        df=df,
        feature_cols=feature_cols,
        medians=medians,
        scaler=scaler,
        feats_imputed=feats_imputed,
        X_scaled=X_scaled,
        nn=nn,
        excel_offset=excel_offset,
        label_col=label_col,
        pos_indices=pos_indices,
        neg_indices=neg_indices,
        pos_X=pos_X,
        neg_X=neg_X,
        pos_nn=pos_nn,
        neg_nn=neg_nn,
    )

In [37]:
def closest_indices_for_row(
    row_idx: int,
    state: KNNState,
    k: int = 5,
    return_distances: bool = False,
) -> Union[
    Tuple[np.ndarray, np.ndarray],
    Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    Dict[str, np.ndarray]
]:
    """
    Return the k closest neighbors for the given row, separately for:
      - positives:  state.label_col == 1  (Outcome==1)
      - negatives:  state.label_col == 0  (Outcome==0)
    Excludes the row itself from its own class subset.
    """
    q = state.X_scaled[row_idx].reshape(1, -1)

    def _query_subset(sub_nn, sub_indices, k, drop_self: bool) -> Tuple[np.ndarray, np.ndarray]:
        if sub_nn is None or len(sub_indices) == 0:
            return np.array([], dtype=int), np.array([], dtype=float)

        n_needed = min(k + (1 if drop_self else 0), len(sub_indices))
        distances, indices = sub_nn.kneighbors(q, n_neighbors=n_needed)
        inds_sub = indices[0].tolist()
        dists = distances[0].tolist()

        global_inds = [int(sub_indices[i]) for i in inds_sub]

        if drop_self and row_idx in global_inds:
            j = global_inds.index(row_idx)
            global_inds.pop(j)
            dists.pop(j)

        global_inds = np.array(global_inds[:k], dtype=int)
        dists = np.array(dists[:k], dtype=float)
        return global_inds, dists

    in_pos = row_idx in set(state.pos_indices.tolist())
    in_neg = row_idx in set(state.neg_indices.tolist())

    pos_inds, pos_dists = _query_subset(state.pos_nn, state.pos_indices, k, drop_self=in_pos)
    neg_inds, neg_dists = _query_subset(state.neg_nn, state.neg_indices, k, drop_self=in_neg)

    if return_distances:
        return pos_inds, pos_dists, neg_inds, neg_dists
    else:
        return pos_inds, neg_inds

In [38]:
def pretty_print_row(row_idx: int, state: KNNState):
    """Prints the full Excel row (all columns) for the given row index."""
    row_df = pd.DataFrame([state.df.iloc[row_idx]], columns=state.df.columns)
    print(row_df.to_string(index=False))

In [39]:
#format the few shot examples
def format_fewshot(row_idx: int, state: KNNState) -> dict:
    """
    Few-shot object for Pima Diabetes:
      - features: dict of state.feature_cols
      - label:    value from state.label_col ('Outcome')
    """
    row = state.df.iloc[row_idx]
    return {
        "patient_id": int(row_idx),
        "features": row[state.feature_cols].to_dict(),
        "label": row[state.label_col],
    }

In [40]:


def render_few_shots_block(df: pd.DataFrame, neighbor_indices: list[int], state: KNNState) -> str:
    """
    Build few-shot text block with lines:
      features_json: {...}
      label: <0 or 1>
    """
    lines = []
    for idx in neighbor_indices:
        ex = format_fewshot(idx, state)
        features_json = json.dumps(ex["features"], ensure_ascii=False)
        lines.append(
            f"patient_id: {ex['patient_id']}\n"
            f"features_json: {features_json}\n"
            f"label: {ex['label']}\n"
        )
    return "\n".join(lines).strip()

In [63]:
#all configurations

#EXCEL_PATH = "../data/Patients_toy.xlsx"       # adjust path if neede
EXCEL_PATH = "../data/diabetes.csv"
SHEET = 0                                           # or a sheet name
MODEL = "gpt-4o-mini"   # any OpenAI chat model
TEMPERATURE = 0.0                                   # deterministic classification
TARGET_COL = "Outcome"                               # ground-truth label in your file
OUT_CSV = "../outputs/Llama8B-Diabetes-CRAIL.csv"

In [56]:
# OpenAI API access: put OPENAI_API_KEY in your environment or in a local .env file
# (see .env.example). Never commit your key.
load_dotenv(find_dotenv())
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set")

In [22]:
# strcutured output and getting the prompt


class IecHsPrediction(BaseModel):
    summary: str = Field(
        ..., description="summary of combinations of feature values that leads to diabetes outcome in the given examples. Compare the given examples to arrive at this crisp summary."
    )
    rationale: str = Field(
        ..., description="Based on the generated summary and given patient's features, key supporting evidence for diabetes outcome to be positive as well as contradicting factors. Followed by overall reasoning for what should be the prediction for diabetes outcome."
    )
    outcome: Literal[0, 1] = Field(..., description="Predicted diabetes outcome class label (0 or 1).")
    confidence: float = Field(
        ..., ge=0.0, le=1.0,
        description="Model self-reported confidence between 0 and 1."
    )
   

#parser = JsonOutputParser(pydantic_object=IecHsPrediction)
parser = PydanticOutputParser(pydantic_object=IecHsPrediction)

system_msg = """You are a careful clinical classifier.  
Your task is - given a single patient's anonymized features, predict whether diabetes outcome is 0 or 1.
You are also given a set of close by example patients and their diabetes outcome scores.
You should first create a summary by comparing the features of given examples to determine which combination of features lead to what value of diabetes outcome scores.
You should then use the generated summary and the given patient's feature to generate a rationale and then predict the diabetes outcome score. 


Output MUST be valid JSON matching this schema:
- summary: A summary (3-5 sentences only) of combinations of feature values that leads to diabetes outcome score in given examples. Compare the given training examples to arrive at this crisp summary.
- rationale: short reasoning for prediction (3-5 sentences). Use the generated summary of the given training examples and given patient's features to come up with rationale for the patient's outcome score.
- outcome: prediction of diabetes outcome 0 or 1 for the given patient.
- confidence: a number in [0, 1]
"""

human_template = """
### Few-shot examples
{few_shots_block}

### Patient to classify
Patient features (key → value) as JSON:
{patient_json}

Instructions:
1) Generate summary of findings given the examples containing the combinations of feature values that leads to diabetes outcome score of 0 or 1. Compare the given training examples to arrive at this crisp summary.
2) Predict outcome (0 or 1). 
3) Return ONLY JSON, no extra text.

Use this format:
{format_instructions}

You should predict diabeted outcome to be 1 only when you are confident else output 0.
Generate output in JSON ONLY.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_msg), ("human", human_template)]
)

# IMPORTANT: inject the raw JSON schema as a *variable*, not as literal text
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

In [57]:
# strcutured output and getting the prompt

class IecHsPrediction(BaseModel):
    rationale: str = Field(
        ..., description="key supporting evidence for outcome to be positive as well as contradicting factors. Followed by overall reasoning for what should be the prediction for dibetest Outcome."
    )
    outcome: Literal[0, 1] = Field(..., description="Predicted outcome class label (0 or 1).")
    confidence: float = Field(
        ..., ge=0.0, le=1.0,
        description="Model self-reported confidence between 0 and 1."
    )
   

#parser = JsonOutputParser(pydantic_object=IecHsPrediction)
parser = PydanticOutputParser(pydantic_object=IecHsPrediction)

system_msg = """You are a careful clinical classifier. 
Given a single patient's anonymized features, predict whether diabetes is 0 or 1.
Only use the provided features—do not invent data.

Output MUST be valid JSON matching this schema:
- rationale: short reasoning for prediction (3-5 sentences)
- outcome: 0 or 1
- confidence: a number in [0, 1]
"""

human_template = """
### Few-shot examples
{few_shots_block}

### Patient to classify
Patient features (key → value) as JSON:
{patient_json}

Instructions:
1) Consider all the supporting as well as contradicting evidences and reason about what the prediction of diabetes should be.
2) Predict outcome (0 or 1).
3) Return ONLY JSON, no extra text.

Use this format:
{format_instructions}

Output MUST be valid JSON matching this schema:
- rationale: short reasoning for prediction (3-5 sentences)
- outcome: 0 or 1
- confidence: a number in [0, 1]
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_msg), ("human", human_template)]
)

# IMPORTANT: inject the raw JSON schema as a *variable*, not as literal text
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

In [58]:
llm = ChatOpenAI(
        model=MODEL,
        temperature=0,
        max_retries=2,
        max_tokens=4096,
        api_key=api_key,
    )

In [59]:
chain = prompt | llm | parser

In [60]:
df = pd.read_csv(EXCEL_PATH)
#selected_cols = ["CRP", "Diagnosis", "Ferritin", "Tocilizumab given"]
#state = build_state(df, use_cols=selected_cols)
state = build_state(df)  # uses first 24 columns (A..X) by default
test_row = 12
pos_inds, pos_dists, neg_inds, neg_dists = closest_indices_for_row(test_row, state, k=3, return_distances=True)
nn_inds = np.concatenate([pos_inds, neg_inds])
all_dists = np.concatenate([pos_dists, neg_dists])
order = np.argsort(all_dists)
nn_inds_sorted = nn_inds[order]
print(render_few_shots_block(df, nn_inds_sorted, state))

patient_id: 259
features_json: {"Pregnancies": 11.0, "Glucose": 155.0, "BloodPressure": 76.0, "SkinThickness": 28.0, "Insulin": 150.0, "BMI": 33.3, "DiabetesPedigreeFunction": 1.353, "Age": 51.0}
label: 1.0

patient_id: 618
features_json: {"Pregnancies": 9.0, "Glucose": 112.0, "BloodPressure": 82.0, "SkinThickness": 24.0, "Insulin": 0.0, "BMI": 28.2, "DiabetesPedigreeFunction": 1.282, "Age": 50.0}
label: 1.0

patient_id: 330
features_json: {"Pregnancies": 8.0, "Glucose": 118.0, "BloodPressure": 72.0, "SkinThickness": 19.0, "Insulin": 0.0, "BMI": 23.1, "DiabetesPedigreeFunction": 1.476, "Age": 46.0}
label: 0.0

patient_id: 152
features_json: {"Pregnancies": 9.0, "Glucose": 156.0, "BloodPressure": 86.0, "SkinThickness": 28.0, "Insulin": 155.0, "BMI": 34.3, "DiabetesPedigreeFunction": 1.189, "Age": 42.0}
label: 1.0

patient_id: 299
features_json: {"Pregnancies": 8.0, "Glucose": 112.0, "BloodPressure": 72.0, "SkinThickness": 0.0, "Insulin": 0.0, "BMI": 23.6, "DiabetesPedigreeFunction": 0.8

In [76]:
results = []
errored_rows = []

state = build_state(df)  # uses first 24 columns (A..X) by default


#selected_cols = ["CRP", "Diagnosis", "Ferritin", "Tocilizumab given"]

#state = build_state(df, use_cols=selected_cols)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring patients"):
    # --- everything before model call runs once ---
    pos_inds, pos_dists, neg_inds, neg_dists = closest_indices_for_row(
        idx, state, k=3, return_distances=True
    )
    nn_inds = np.concatenate([pos_inds, neg_inds])
    all_dists = np.concatenate([pos_dists, neg_dists])
    order = np.argsort(all_dists)
    nn_inds_sorted = nn_inds[order]

    few_shot_block = render_few_shots_block(df, nn_inds_sorted, state)
    record = format_fewshot(idx, state)
    patient_features = record["features"]

    # safety
    assert TARGET_COL not in patient_features

    # --- retry loop ONLY around the model call ---
    success = False
    last_error = None
    for attempt in range(5):
        try:
            pred: IecHsPrediction = chain.invoke({
                "few_shots_block": few_shot_block,
                "patient_json": json.dumps(patient_features, ensure_ascii=False)
            })

            
            results.append({
                "row_index": idx,
                "outcome": int(pred.outcome),
                "confidence": float(pred.confidence),
                "rationale": pred.rationale,
                "iec_hs_true": row.get(TARGET_COL, None)
            })

            print(f"prediction for row {idx} is {pred.outcome} with confidence {pred.confidence} and true label is {row.get(TARGET_COL, None)}")
            #print(f"summary for row {idx} is {pred.summary}")
            #print(f"rationale for row {idx} is {pred.rationale}")


            success = True
            break  # exit retry loop if successful
        except Exception as e:
            last_error = e
            print(f"Attempt {attempt+1} failed for row {idx}: {e}")
            print(f"output is {pred}")
            continue  # try again

    if not success:
        errored_rows.append((idx, str(last_error)))
        print(f"prediction for row {idx} is SKIPPED after 5 attempts")

Scoring patients:   0%|          | 1/768 [00:02<29:52,  2.34s/it]

prediction for row 0 is 1 with confidence 0.6 and true label is 1.0


Scoring patients:   0%|          | 2/768 [00:04<27:02,  2.12s/it]

prediction for row 1 is 0 with confidence 0.7 and true label is 0.0


Scoring patients:   0%|          | 3/768 [00:06<25:06,  1.97s/it]

...

prediction for row 765 is 1 with confidence 0.7 and true label is 0.0


Scoring patients: 100%|█████████▉| 767/768 [24:04<00:01,  1.76s/it]

prediction for row 766 is 1 with confidence 0.7 and true label is 1.0


Scoring patients: 100%|██████████| 768/768 [24:06<00:00,  1.88s/it]

prediction for row 767 is 0 with confidence 0.7 and true label is 0.0


In [77]:
out_df = pd.DataFrame(results)

# Keep only rows where both are non-null
valid = out_df["outcome"].notna() & out_df["iec_hs_true"].notna()
y_true = out_df.loc[valid, "iec_hs_true"].astype(int)
y_pred = out_df.loc[valid, "outcome"].astype(int)

# Overall metrics (positive class = 1)
accuracy   = accuracy_score(y_true, y_pred)
precision1 = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
recall1    = recall_score(y_true, y_pred,    pos_label=1, zero_division=0)
f1_pos     = f1_score(y_true, y_pred,        pos_label=1, zero_division=0)

# Negative-class metrics (label = 0)
precision0 = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
recall0    = recall_score(y_true, y_pred,    pos_label=0, zero_division=0)
f1_neg     = f1_score(y_true, y_pred,        pos_label=0, zero_division=0)

# Macro F1 (unweighted mean of per-class F1s)
f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"Accuracy:     {accuracy:.3f}")
print(f"Pos P/R/F1:   P={precision1:.3f}  R={recall1:.3f}  F1={f1_pos:.3f}")
print(f"Neg P/R/F1:   P={precision0:.3f}  R={recall0:.3f}  F1={f1_neg:.3f}")
print(f"Macro F1:     {f1_macro:.3f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print("\nConfusion Matrix (rows = true, cols = pred):")
print("         Pred 0   Pred 1")
print(f"True 0   {cm[0,0]:6}   {cm[0,1]:6}")
print(f"True 1   {cm[1,0]:6}   {cm[1,1]:6}")

Accuracy:     0.665
Pos P/R/F1:   P=0.514  R=0.780  F1=0.619
Neg P/R/F1:   P=0.837  R=0.604  F1=0.702
Macro F1:     0.660

Confusion Matrix (rows = true, cols = pred):
         Pred 0   Pred 1
True 0      302      198
True 1       59      209


In [65]:
out_df = pd.DataFrame(results)
print(out_df.columns.tolist())


#print(out_df)
out_df.to_csv(OUT_CSV, index=False)

['row_index', 'outcome', 'confidence', 'rationale', 'iec_hs_true']


In [72]:
THRESH = 0.60  # <= 0.60 => flip 1 -> 0

out_df = pd.DataFrame(results)

# Require prediction, truth, and confidence
if not {"outcome", "iec_hs_true", "confidence"}.issubset(out_df.columns):
    raise ValueError(f"Expected columns 'outcome', 'iec_hs_true', 'confidence'. Got: {list(out_df.columns)}")

# Keep rows with all needed fields
valid = out_df["outcome"].notna() & out_df["iec_hs_true"].notna() & out_df["confidence"].notna()
y_true = out_df.loc[valid, "iec_hs_true"].astype(int).to_numpy()
y_pred = out_df.loc[valid, "outcome"].astype(int).to_numpy()
conf   = out_df.loc[valid, "confidence"].astype(float).to_numpy()

# Flip predicted 1s with low confidence to 0
y_pred_adj = np.where((y_pred == 1) & (conf <= THRESH), 0, y_pred)

# Metrics with adjusted predictions
accuracy   = accuracy_score(y_true, y_pred_adj)
precision1 = precision_score(y_true, y_pred_adj, pos_label=1, zero_division=0)
recall1    = recall_score(y_true, y_pred_adj,    pos_label=1, zero_division=0)
f1_pos     = f1_score(y_true, y_pred_adj,        pos_label=1, zero_division=0)

precision0 = precision_score(y_true, y_pred_adj, pos_label=0, zero_division=0)
recall0    = recall_score(y_true, y_pred_adj,    pos_label=0, zero_division=0)
f1_neg     = f1_score(y_true, y_pred_adj,        pos_label=0, zero_division=0)

f1_macro   = f1_score(y_true, y_pred_adj, average="macro", zero_division=0)

print(f"Applied confidence rule: if pred==1 and confidence <= {THRESH:.2f} -> set pred=0")
print(f"Samples evaluated: {len(y_true)}")
print(f"Accuracy:     {accuracy:.3f}")
print(f"Pos P/R/F1:   P={precision1:.3f}  R={recall1:.3f}  F1={f1_pos:.3f}")
print(f"Neg P/R/F1:   P={precision0:.3f}  R={recall0:.3f}  F1={f1_neg:.3f}")
print(f"Macro F1:     {f1_macro:.3f}")

cm = confusion_matrix(y_true, y_pred_adj, labels=[0, 1])
print("\nConfusion Matrix (rows = true, cols = pred):")
print("         Pred 0   Pred 1")
print(f"True 0   {cm[0,0]:6}   {cm[0,1]:6}")
print(f"True 1   {cm[1,0]:6}   {cm[1,1]:6}")

Applied confidence rule: if pred==1 and confidence <= 0.60 -> set pred=0
Samples evaluated: 768
Accuracy:     0.688
Pos P/R/F1:   P=0.548  R=0.597  F1=0.571
Neg P/R/F1:   P=0.773  R=0.736  F1=0.754
Macro F1:     0.663

Confusion Matrix (rows = true, cols = pred):
         Pred 0   Pred 1
True 0      368      132
True 1      108      160


In [64]:
out_df = pd.DataFrame(results)
print(out_df.columns.tolist())


#print(out_df)
out_df.to_csv(OUT_CSV, index=False)

['row_index', 'outcome', 'confidence', 'rationale', 'iec_hs_true']


In [167]:
len(errored_rows)

14